<a href="https://colab.research.google.com/github/Erickpython/kodeCamp_5X-MachineLearning/blob/main/Next_Word_Prediction_from_IMDB_Dataset_using_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Imports**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

## **Load IMDB Dataset**

In [ ]:
dataset = load_dataset("imdb")

train_texts = dataset["train"]["text"]

# Reduce size for faster demo
train_texts = train_texts[:1000]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## **Simple Tokenizer**

In [ ]:
def tokenize(text):
    return text.lower().split()

In [ ]:
from IPython.display import Markdown

In [ ]:
Markdown("- " + "\n\n- ".join(train_texts[:10]))

- I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, even then it's not shot like some cheaply made porno. While my countrymen mind find it shocking, in reality sex and nudity are a major staple in Swedish cinema. Even Ingmar Bergman, arguably their answer to good old boy John Ford, had sex scenes in his films.<br /><br />I do commend the filmmakers for the fact that any sex shown in the film is shown for artistic purposes rather than just to shock people and make money to be shown in pornographic theaters in America. I AM CURIOUS-YELLOW is a good film for anyone wanting to study the meat and potatoes (no pun intended) of Swedish cinema. But really, this film doesn't have much of a plot.

- "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don't exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris in sight. And those pretentious indie movies like The Brown Bunny, in which we're treated to the site of Vincent Gallo's throbbing johnson, but not a trace of pink visible on Chloe Sevigny. Before crying (or implying) "double-standard" in matters of nudity, the mentally obtuse should take into account one unavoidably obvious anatomical difference between men and women: there are no genitals on display when actresses appears nude, and the same cannot be said for a man. In fact, you generally won't see female genitals in an American film in anything short of porn or explicit erotica. This alleged double-standard is less a double standard than an admittedly depressing ability to come to terms culturally with the insides of women's bodies.

- If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />

- This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impressive, undeservedly good, photo. Apart from that, what strikes me most is the endless stream of silliness. Lena Nyman has to be most annoying actress in the world. She acts so stupid and with all the nudity in this film,...it's unattractive. Comparing to Godard's film, intellectuality has been replaced with stupidity. Without going too far on this subject, I would say that follows from the difference in ideals between the French and the Swedish society.<br /><br />A movie of its time, and place. 2/10.

- Oh, brother...after hearing about this ridiculous film for umpteen years all I can think of is that old Peggy Lee song..<br /><br />"Is that all there is??" ...I was just an early teen when this smoked fish hit the U.S. I was too young to get in the theater (although I did manage to sneak into "Goodbye Columbus"). Then a screening at a local film museum beckoned - Finally I could see this film, except now I was as old as my parents were when they schlepped to see it!!<br /><br />The ONLY reason this film was not condemned to the anonymous sands of time was because of the obscenity case sparked by its U.S. release. MILLIONS of people flocked to this stinker, thinking they were going to see a sex film...Instead, they got lots of closeups of gnarly, repulsive Swedes, on-street interviews in bland shopping malls, asinie political pretension...and feeble who-cares simulated sex scenes with saggy, pale actors.<br /><br />Cultural icon, holy grail, historic artifact..whatever this thing was, shred it, burn it, then stuff the ashes in a lead box!<br /><br />Elite esthetes still scrape to find value in its boring pseudo revolutionary political spewings..But if it weren't for the censorship scandal, it would have been ignored, then forgotten.<br /><br />Instead, the "I Am Blank, Blank" rhythymed title was repeated endlessly for years as a titilation for porno films (I am Curious, Lavender - for gay films, I Am Curious, Black - for blaxploitation films, etc..) and every ten years or so the thing rises from the dead, to be viewed by a new generation of suckers who want to see that "naughty sex film" that "revolutionized the film industry"...<br /><br />Yeesh, avoid like the plague..Or if you MUST see it - rent the video and fast forward to the "dirty" parts, just to get it over with.<br /><br />

- I would put this at the top of my list of films in the category of unwatchable trash! There are films that are bad, but the worst kind are the ones that are unwatchable but you are suppose to like them because they are supposed to be good for you! The sex sequences, so shocking in its day, couldn't even arouse a rabbit. The so called controversial politics is strictly high school sophomore amateur night Marxism. The film is self-consciously arty in the worst sense of the term. The photography is in a harsh grainy black and white. Some scenes are out of focus or taken from the wrong angle. Even the sound is bad! And some people call this art?<br /><br />

- Whoever wrote the screenplay for this movie obviously never consulted any books about Lucille Ball, especially her autobiography. I've never seen so many mistakes in a biopic, ranging from her early years in Celoron and Jamestown to her later years with Desi. I could write a whole list of factual errors, but it would go on for pages. In all, I believe that Lucille Ball is one of those inimitable people who simply cannot be portrayed by anyone other than themselves. If I were Lucie Arnaz and Desi, Jr., I would be irate at how many mistakes were made in this film. The filmmakers tried hard, but the movie seems awfully sloppy to me.

- When I first saw a glimpse of this movie, I quickly noticed the actress who was playing the role of Lucille Ball. Rachel York's portrayal of Lucy is absolutely awful. Lucille Ball was an astounding comedian with incredible talent. To think about a legend like Lucille Ball being portrayed the way she was in the movie is horrendous. I cannot believe out of all the actresses in the world who could play a much better Lucy, the producers decided to get Rachel York. She might be a good actress in other roles but to play the role of Lucille Ball is tough. It is pretty hard to find someone who could resemble Lucille Ball, but they could at least find someone a bit similar in looks and talent. If you noticed York's portrayal of Lucy in episodes of I Love Lucy like the chocolate factory or vitavetavegamin, nothing is similar in any way-her expression, voice, or movement.<br /><br />To top it all off, Danny Pino playing Desi Arnaz is horrible. Pino does not qualify to play as Ricky. He's small and skinny, his accent is unreal, and once again, his acting is unbelievable. Although Fred and Ethel were not similar either, they were not as bad as the characters of Lucy and Ricky.<br /><br />Overall, extremely horrible casting and the story is badly told. If people want to understand the real life situation of Lucille Ball, I suggest watching A&E Biography of Lucy and Desi, read the book from Lucille Ball herself, or PBS' American Masters: Finding Lucy. If you want to see a docudrama, "Before the Laughter" would be a better choice. The casting of Lucille Ball and Desi Arnaz in "Before the Laughter" is much better compared to this. At least, a similar aspect is shown rather than nothing.

- Who are these "They"- the actors? the filmmakers? Certainly couldn't be the audience- this is among the most air-puffed productions in existence. It's the kind of movie that looks like it was a lot of fun to shoot TOO much fun, nobody is getting any actual work done, and that almost always makes for a movie that's no fun to watch.<br /><br />Ritter dons glasses so as to hammer home his character's status as a sort of doppleganger of the bespectacled Bogdanovich; the scenes with the breezy Ms. Stratten are sweet, but have an embarrassing, look-guys-I'm-dating-the-prom-queen feel to them. Ben Gazzara sports his usual cat's-got-canary grin in a futile attempt to elevate the meager plot, which requires him to pursue Audrey Hepburn with all the interest of a narcoleptic at an insomnia clinic. In the meantime, the budding couple's respective children (nepotism alert: Bogdanovich's daughters) spew cute and pick up some fairly disturbing pointers on 'love' while observing their parents. (Ms. Hepburn, drawing on her dignity, manages to rise above the proceedings- but she has the monumental challenge of playing herself, ostensibly.) Everybody looks great, but so what? It's a movie and we can expect that much, if that's what you're looking for you'd be better off picking up a copy of Vogue.<br /><br />Oh- and it has to be mentioned that Colleen Camp thoroughly annoys, even apart from her singing, which, while competent, is wholly unconvincing... the country and western numbers are woefully mismatched with the standards on the soundtrack. Surely this is NOT what Gershwin (who wrote the song from which the movie's title is derived) had in mind; his stage musicals of the 20's may have been slight, but at least they were long on charm. "They All Laughed" tries to coast on its good intentions, but nobody- least of all Peter Bogdanovich - has the good sense to put on the brakes.<br /><br />Due in no small part to the tragic death of Dorothy Stratten, this movie has a special place in the heart of Mr. Bogdanovich- he even bought it back from its producers, then distributed it on his own and went bankrupt when it didn't prove popular. His rise and fall is among the more sympathetic and tragic of Hollywood stories, so there's no joy in criticizing the film... there _is_ real emotional investment in Ms. Stratten's scenes. But "Laughed" is a faint echo of "The Last Picture Show", "Paper Moon" or "What's Up, Doc"- following "Daisy Miller" and "At Long Last Love", it was a thundering confirmation of the phase from which P.B. has never emerged.<br /><br />All in all, though, the movie is harmless, only a waste of rental. I want to watch people having a good time, I'll go to the park on a sunny day. For filmic expressions of joy and love, I'll stick to Ernest Lubitsch and Jaques Demy...

- This is said to be a personal film for Peter Bogdonavitch. He based it on his life but changed things around to fit the characters, who are detectives. These detectives date beautiful models and have no problem getting them. Sounds more like a millionaire playboy filmmaker than a detective, doesn't it? This entire movie was written by Peter, and it shows how out of touch with real people he was. You're supposed to write what you know, and he did that, indeed. And leaves the audience bored and confused, and jealous, for that matter. This is a curio for people who want to see Dorothy Stratten, who was murdered right after filming. But Patti Hanson, who would, in real life, marry Keith Richards, was also a model, like Stratten, but is a lot better and has a more ample part. In fact, Stratten's part seemed forced; added. She doesn't have a lot to do with the story, which is pretty convoluted to begin with. All in all, every character in this film is somebody that very few people can relate with, unless you're millionaire from Manhattan with beautiful supermodels at your beckon call. For the rest of us, it's an irritating snore fest. That's what happens when you're out of touch. You entertain your few friends with inside jokes, and bore all the rest.

## **Build Vocabulary**

In [ ]:
from collections import Counter

counter = Counter()

for text in train_texts:
    tokens = tokenize(text)
    counter.update(tokens)

# Keep most common words
vocab_size = 20000
most_common = counter.most_common(vocab_size - 2)

stoi = {"<pad>": 0, "<unk>": 1}
for i, (word, _) in enumerate(most_common, start=2):
    stoi[word] = i

itos = {i: s for s, i in stoi.items()}

In [ ]:
def encode(text):
    tokens = tokenize(text)
    return [stoi.get(token, stoi["<unk>"]) for token in tokens]

Create Language Modeling Dataset

We train for next word prediction.

Input:
```
The movie was
```

```
movie was great
```

In [ ]:
class IMDBLanguageDataset(Dataset):
    def __init__(self, texts, seq_len=50):
        self.seq_len = seq_len
        self.data = []

        for text in texts:
            encoded = encode(text)

            for i in range(len(encoded) - seq_len):
                x = encoded[i:i+seq_len]
                y = encoded[i+1:i+seq_len+1]
                self.data.append((x, y))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x), torch.tensor(y)

In [ ]:
seq_len = 50
dataset = IMDBLanguageDataset(train_texts, seq_len)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

In [ ]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.transformer = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_mask(self, size):
        mask = torch.triu(torch.ones(size, size), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask

    def forward(self, x):
        seq_len = x.size(1)

        mask = self.generate_mask(seq_len).to(x.device)

        emb = self.embedding(x)
        emb = self.pos_encoding(emb)

        output = self.transformer(
            emb,
            emb,
            tgt_mask=mask
        )

        logits = self.fc_out(output)
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerLM(len(stoi)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

In [ ]:
for x, y in loader:
    print(x)
    print(y)
    break

tensor([[16748,     7,     3,  ...,  1400,    16,  3178],
        [  233, 16339,     4,  ...,   440, 16346,   220],
        [    2,  1059, 16623,  ...,   302,    18,     2],
        ...,
        [    1,    46,    16,  ...,    24,     1,    17],
        [   10,   479,     2,  ...,    75,  3125,    20],
        [ 2434,    78,    57,  ...,     3,   633,  3454]])
tensor([[    7,     3,   751,  ...,    16,  3178,  7755],
        [16339,     4, 16340,  ..., 16346,   220,    73],
        [ 1059, 16623,    13,  ...,    18,     2,   638],
        ...,
        [   46,    16,    44,  ...,     1,    17,    18],
        [  479,     2, 10327,  ...,  3125,    20,  5490],
        [   78,    57,  1703,  ...,   633,  3454,    50]])


In [ ]:
from tqdm import tqdm

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    batch = 0
    for batch, (x, y) in enumerate(tqdm(loader, f"Epoch: {epoch}")):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

Epoch: 0: 100%|██████████| 5715/5715 [04:51<00:00, 19.61it/s]


Epoch 1, Loss: 0.9645


Epoch: 1: 100%|██████████| 5715/5715 [04:50<00:00, 19.67it/s]


Epoch 2, Loss: 0.1290


Epoch: 2: 100%|██████████| 5715/5715 [04:51<00:00, 19.60it/s]


Epoch 3, Loss: 0.1097


Epoch: 3: 100%|██████████| 5715/5715 [04:50<00:00, 19.68it/s]


Epoch 4, Loss: 0.0982


Epoch: 4: 100%|██████████| 5715/5715 [04:50<00:00, 19.67it/s]

Epoch 5, Loss: 0.0890


In [ ]:
def generate_text(model, start_text, max_len=50):
    model.eval()

    tokens = encode(start_text)
    tokens = torch.tensor(tokens).unsqueeze(0).to(device)

    for _ in range(max_len):
        with torch.no_grad():
            logits = model(tokens)

        next_token = torch.argmax(logits[:, -1, :], dim=-1)

        tokens = torch.cat([tokens, next_token.unsqueeze(0)], dim=1)

    result = [itos[token.item()] for token in tokens[0]]
    return " ".join(result)

In [ ]:
print(generate_text(model, "this movie is not"))

this movie is not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not not


In [ ]:
model.eval()

tokens = encode("this movie is")
tokens = torch.tensor(tokens).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(tokens)

print(logits.shape)
next_token = torch.argmax(logits[:, -1, :], dim=-1)

torch.Size([1, 3, 20000])


In [ ]:
[itos[i.item()] for i in torch.topk(logits, 10).indices[:, -3, :][0]]

['movie', 'is', 'this', 'movie,', ')', 'it', 'plot', '/><br', 'too', 'one']

In [ ]:
[itos[i.item()] for i in torch.topk(logits, 10).indices[:, -2, :][0]]

['is',
 'does',
 'nastie',
 'danza',
 'are',
 'and',
 'cesspool',
 'was',
 'this',
 'of']

In [ ]:
[itos[i.item()] for i in torch.topk(logits, 10).indices[:, -1, :][0]]

['is',
 'movie',
 'plot',
 'faith',
 'it',
 'output',
 'danza',
 'pound,',
 ')',
 'slash']

In [ ]:
logits

tensor([[[-14.1885,   5.1923,   6.3700,  ...,  -8.8460, -11.2849,  -5.5579],
         [ -7.1193,   4.9775,   4.1082,  ...,  -2.5789, -10.6600,  -5.9686],
         [ -7.1954,   4.6465,   3.3456,  ...,  -3.3123, -12.5907,  -3.1805]]],
       device='cuda:0')